## Condition Configuration

- `CONDITION`: condition name whose rephrased proposals should be reviewed.
- `SKIP_NCEMS_IF_EXISTS`: reuse the latest NCEMS review JSON if it already exists.
- `SKIP_NOVELTY_IF_EXISTS`: reuse the latest novelty review JSON if it already exists.
- `SKIP_REPHRASED_AI_REVIEWS_IF_EXISTS`: reuse the rephrased AI NCEMS review JSON when it matches the latest NCEMS review source.
- `REBUILD_REVIEW_SCORES`: rebuild `review_scores_wide.csv` from the latest NCEMS review file.


In [ ]:
# Auto-managed by src/run_condition.py
CONDITION = 'minimal'
SKIP_NCEMS_IF_EXISTS = True
SKIP_NOVELTY_IF_EXISTS = True
SKIP_REPHRASED_AI_REVIEWS_IF_EXISTS = True
REBUILD_REVIEW_SCORES = True


# Generate AI Reviews

This notebook generates both NCEMS-criteria reviews and novelty reviews for a single experimental condition, then builds `review_scores_wide.csv` for downstream analysis.

Run this notebook after `compare_proposals_rephrased.ipynb`, because novelty review generation depends on `proposal_lit_neighbors.json`.

## Condition Configuration

In [ ]:
CONDITION = 'minimal'
SKIP_NCEMS_IF_EXISTS = True
SKIP_NOVELTY_IF_EXISTS = True
SKIP_REPHRASED_AI_REVIEWS_IF_EXISTS = True
REBUILD_REVIEW_SCORES = True


## Setup and Path Checks

In [ ]:
import json
import subprocess
import sys
from pathlib import Path


def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate.resolve()
    raise RuntimeError('Could not find project root containing src/ and data/.')



def is_reusable_review_file(path):
    try:
        with open(path) as f:
            payload = json.load(f)
    except Exception:
        return False

    human_inputs = ' '.join(str(p) for p in payload.get('human_inputs', []) or [])
    ai_input = str(payload.get('ai_input', ''))
    return (
        payload.get('rephrased') is True
        and 'data/human-proposals/rephrased' in human_inputs
        and f'data/ai-proposals/rephrased/{CONDITION}' in ai_input
    )


def latest_reusable_review_file(review_dir, pattern):
    for path in sorted(review_dir.glob(pattern), reverse=True):
        if is_reusable_review_file(path):
            return path
    return None


PROJECT_ROOT = find_project_root()
SCRIPTS_DIR = PROJECT_ROOT / 'src'
AI_REPHRASED_DIR = PROJECT_ROOT / 'data' / 'ai-proposals' / 'rephrased' / CONDITION
HUMAN_REPHRASED_DIR = PROJECT_ROOT / 'data' / 'human-proposals' / 'rephrased'
LIT_NEIGHBORS_PATH = PROJECT_ROOT / 'results' / 'tables' / 'rephrased' / CONDITION / 'proposal_lit_neighbors.json'
NCEMS_REVIEWS_DIR = PROJECT_ROOT / 'data' / 'reviews' / 'ai_reviews' / CONDITION / 'ncems_criteria'
NOVELTY_REVIEWS_DIR = PROJECT_ROOT / 'data' / 'reviews' / 'ai_reviews' / CONDITION / 'novelty'
REVIEW_SCORES_PATH = PROJECT_ROOT / 'results' / 'tables' / 'rephrased' / CONDITION / 'review_scores_wide.csv'
AI_REPHRASED_REVIEWS_PATH = NCEMS_REVIEWS_DIR / 'rephrased' / 'ncems_reviews_rephrased.json'


def resolve_project_path(path):
    p = Path(str(path))
    return p if p.is_absolute() else PROJECT_ROOT / p


def rephrased_ai_reviews_current(rephrased_path, source_path):
    try:
        with open(rephrased_path) as f:
            payload = json.load(f)
    except Exception:
        return False

    source_file = payload.get('source_file', '')
    if not source_file:
        return False
    return resolve_project_path(source_file).resolve() == Path(source_path).resolve()

print(f'Condition: {CONDITION}')
print(f'Project root: {PROJECT_ROOT}')
print(f'AI rephrased dir: {AI_REPHRASED_DIR}')
print(f'Human rephrased dir: {HUMAN_REPHRASED_DIR}')
print(f'Literature neighbors path: {LIT_NEIGHBORS_PATH}')
print(f'AI rephrased NCEMS reviews path: {AI_REPHRASED_REVIEWS_PATH}')

if not AI_REPHRASED_DIR.exists():
    raise FileNotFoundError(f'Missing AI rephrased proposals directory: {AI_REPHRASED_DIR}')
if not HUMAN_REPHRASED_DIR.exists():
    raise FileNotFoundError(f'Missing human rephrased proposals directory: {HUMAN_REPHRASED_DIR}')
if not LIT_NEIGHBORS_PATH.exists():
    raise FileNotFoundError(
        f'Missing literature neighbors file: {LIT_NEIGHBORS_PATH}. '
        'Run compare_proposals_rephrased.ipynb first.'
    )

print('✓ Required proposal and literature-neighbor inputs are present')


## Generate NCEMS-Criteria Reviews

In [ ]:
latest_ncems = latest_reusable_review_file(NCEMS_REVIEWS_DIR, 'ncems_reviews_*.json')
if latest_ncems and SKIP_NCEMS_IF_EXISTS:
    print(f'✓ Reusing existing NCEMS reviews: {latest_ncems}')
else:
    subprocess.check_call([
        sys.executable,
        str(SCRIPTS_DIR / 'generate_reviews_ncems_criteria.py'),
        '--condition',
        CONDITION,
    ], cwd=PROJECT_ROOT)

    latest_ncems = sorted(NCEMS_REVIEWS_DIR.glob('ncems_reviews_*.json'))[-1]
    print(f'✓ NCEMS reviews written to: {latest_ncems}')


## Rephrase NCEMS AI Reviews


In [ ]:
if AI_REPHRASED_REVIEWS_PATH.exists() and SKIP_REPHRASED_AI_REVIEWS_IF_EXISTS and rephrased_ai_reviews_current(AI_REPHRASED_REVIEWS_PATH, latest_ncems):
    print(f'✓ Reusing rephrased AI NCEMS reviews: {AI_REPHRASED_REVIEWS_PATH}')
else:
    subprocess.check_call([
        sys.executable,
        str(SCRIPTS_DIR / 'rephrase_reviews.py'),
        '--skip-human',
        '--ai-reviews-path',
        str(latest_ncems),
        '--ai-output-path',
        str(AI_REPHRASED_REVIEWS_PATH),
    ], cwd=PROJECT_ROOT)
    print(f'✓ Rephrased AI NCEMS reviews written to: {AI_REPHRASED_REVIEWS_PATH}')


## Generate Novelty Reviews

In [ ]:
latest_novelty = latest_reusable_review_file(NOVELTY_REVIEWS_DIR, 'novelty_reviews_*.json')
if latest_novelty and SKIP_NOVELTY_IF_EXISTS:
    print(f'✓ Reusing existing novelty reviews: {latest_novelty}')
else:
    subprocess.check_call([
        sys.executable,
        str(SCRIPTS_DIR / 'generate_reviews_novelty.py'),
        '--condition',
        CONDITION,
    ], cwd=PROJECT_ROOT)

    latest_novelty = sorted(NOVELTY_REVIEWS_DIR.glob('novelty_reviews_*.json'))[-1]
    print(f'✓ Novelty reviews written to: {latest_novelty}')


## Build Review Score Table

In [ ]:
if REBUILD_REVIEW_SCORES or not REVIEW_SCORES_PATH.exists():
    subprocess.check_call([
        sys.executable,
        str(SCRIPTS_DIR / 'build_review_scores_wide.py'),
        '--condition',
        CONDITION,
    ], cwd=PROJECT_ROOT)
    print(f'✓ Review score table written to: {REVIEW_SCORES_PATH}')
else:
    print(f'✓ Reusing existing review score table: {REVIEW_SCORES_PATH}')


## Output Summary

In [ ]:
summary = {
    'condition': CONDITION,
    'latest_ncems_reviews': str(sorted(NCEMS_REVIEWS_DIR.glob('ncems_reviews_*.json'))[-1]),
    'rephrased_ai_ncems_reviews': str(AI_REPHRASED_REVIEWS_PATH),
    'latest_novelty_reviews': str(sorted(NOVELTY_REVIEWS_DIR.glob('novelty_reviews_*.json'))[-1]),
    'review_scores_wide': str(REVIEW_SCORES_PATH),
}

print(json.dumps(summary, indent=2))
